## This notebook should investigate the numbers per group of the MPRAsnakeflow result count table

### Outline:
- read count table 
- merge over replicates
- get counts per group

In [2]:
import pandas as pd
import numpy as np
import os
import yaml

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf
# # reload helpful_functions
# from importlib import reload
# reload(hf)

config_path = "../../global80K_config.yaml"
# load config file
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

#### Investigating numbers of the design

In [3]:
design_fa = hf.fasta_to_dataframe(config['files']['final_design']['design_fasta'])
design_fa['label'] = design_fa['header'].apply(hf.get_label)
design_fa['label'].value_counts()

label
cardiac_neuro_cava_random            73940
MK                                    2397
C_positive_heart_AB                    909
GC_Selvarajan                          364
GC_Vista                               256
C_negative_heart_MK                    243
C_negative_neuron_MK                   222
C_negative_neuron_NP                   217
GC_Mendelian_variants                  209
GC_Kircher                             203
C_SLEA                                 200
GC_Cort_Chengyu                        185
C_positive_neuron_NP                    99
C_positive_heart_CAD                    97
C_positive_heart_MK                     97
C_positive_neuron_MK                    96
C_positive_neuron_CD                    94
GC_GABA_Chengyu                         85
GC_DNase_positive_shuffeled             55
GC_Atrial_fib                           45
GC_DNase_positive                       41
GC_Glut_Chengyu                         40
GC_Mohlke                               34
GC_DN

#### Investigating numbers of the MPRAsnakeflow result


In [4]:
def combine_replicates(df_allreps, total_dna_counts, total_rna_counts):
    df_allreps = df_allreps.groupby(by=["condition", "name"]).aggregate(
        {
            "replicate": "count",
            "dna_counts": ["sum", "mean"],
            "rna_counts": ["sum", "mean"],
            "dna_normalized": "mean",
            "rna_normalized": "mean",
            "ratio": "mean",
            "log2": "mean",
            "n_obs_bc": ["sum", "mean"],
        }
    )
    
    df_allreps = df_allreps.reset_index()
    df_out = df_allreps.iloc[:, 0:2]
    df_out.columns = ["condition", "name"]

    df_out["replicates"] = df_allreps.replicate["count"]

    scaling = 10**6

    df_out["dna_counts"] = df_allreps.dna_counts["sum"]
    df_out["rna_counts"] = df_allreps.rna_counts["sum"]


    df_out["dna_normalized"] = df_out["dna_counts"] / total_dna_counts * scaling
    df_out["rna_normalized"] = df_out["rna_counts"] / total_rna_counts * scaling

    df_out["ratio"] = df_out["rna_normalized"] / df_out["dna_normalized"]
    df_out["log2"] = np.log2(df_out.ratio)

    df_out["mean_dna_counts"] = df_allreps.dna_counts["mean"]
    df_out["mean_rna_counts"] = df_allreps.rna_counts["mean"]
    df_out["mean_dna_normalized"] = df_allreps.dna_normalized["mean"]
    df_out["mean_rna_normalized"] = df_allreps.rna_normalized["mean"]
    df_out["mean_ratio"] = df_allreps.ratio["mean"]
    df_out["mean_log2"] = df_allreps.log2["mean"]
    
    df_out["mean_n_obs_bc"] = df_allreps.n_obs_bc["mean"].apply(int)
    return df_out

#### Check number of missing and found sequences

In [54]:
# TODO: number of unique sequences with 3 replicates
old_filtering_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/resequencing_results/old_bam_filtering/results/experiments/standard_bwa/assigned_counts/assignmentFixDuplicates/standardConfig/NGN2_allreps_minThreshold_merged.tsv.gz'
mprasnakeflow_count_table = pd.read_csv(old_filtering_path, sep="\t")

# merge the sequences over the replicates
total_dna_counts = mprasnakeflow_count_table[["dna_counts"]].sum().iloc[0]
total_rna_counts = mprasnakeflow_count_table[["rna_counts"]].sum().iloc[0]
combined_mprasnakeflow_count_table = combine_replicates(mprasnakeflow_count_table, total_dna_counts, total_rna_counts)
combined_mprasnakeflow_count_table = combined_mprasnakeflow_count_table.loc[combined_mprasnakeflow_count_table['name'] != 'no_BC']

combined_mprasnakeflow_count_table.loc[combined_mprasnakeflow_count_table['name'] != 'no_BC']['name'].nunique() # 70788
combined_mprasnakeflow_count_table = combined_mprasnakeflow_count_table.loc[combined_mprasnakeflow_count_table['replicates'] == 3]
combined_mprasnakeflow_count_table['label'] = combined_mprasnakeflow_count_table['name'].apply(hf.get_label)
combined_mprasnakeflow_count_table['label'].value_counts()

label
cardiac_neuro_cava_random            64460
MK                                    2031
C_positive_heart_AB                    612
GC_Selvarajan                          323
GC_Vista                               227
C_negative_heart_MK                    199
GC_Kircher                             191
GC_Cort_Chengyu                        185
C_negative_neuron_MK                   179
C_SLEA                                 174
GC_Mendelian_variants                  135
C_negative_neuron_NP                   118
C_positive_heart_CAD                    91
C_positive_neuron_MK                    91
C_positive_heart_MK                     85
C_positive_neuron_CD                    69
GC_GABA_Chengyu                         68
C_positive_neuron_NP                    65
GC_DNase_positive_shuffeled             51
GC_Atrial_fib                           41
GC_DNase_positive                       38
GC_Glut_Chengyu                         31
GC_Mohlke                               30
GC_DN

In [55]:
new_filtering_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/resequencing_results/new_bam_filtering/results/experiments/standard_bwa/assigned_counts/assignmentFixDuplicates/standardConfig/NGN2_allreps_minThreshold_merged.tsv.gz'

new_filtering_mprasnakeflow_count_table = pd.read_csv(new_filtering_path, sep="\t")
new_filtering_mprasnakeflow_count_table = new_filtering_mprasnakeflow_count_table.loc[new_filtering_mprasnakeflow_count_table['name'] != 'no_BC']
# merge the sequences over the replicates
total_dna_counts = new_filtering_mprasnakeflow_count_table[["dna_counts"]].sum().iloc[0]
total_rna_counts = new_filtering_mprasnakeflow_count_table[["rna_counts"]].sum().iloc[0]
combined_new_filtering_mprasnakeflow_count_table = combine_replicates(new_filtering_mprasnakeflow_count_table, total_dna_counts, total_rna_counts)
combined_new_filtering_mprasnakeflow_count_table = combined_new_filtering_mprasnakeflow_count_table.loc[combined_new_filtering_mprasnakeflow_count_table['replicates'] == 3]

# remove no_BC
combined_new_filtering_mprasnakeflow_count_table = combined_new_filtering_mprasnakeflow_count_table.loc[combined_new_filtering_mprasnakeflow_count_table['name'] != 'no_BC']
combined_new_filtering_mprasnakeflow_count_table['name'].nunique()
combined_new_filtering_mprasnakeflow_count_table.loc[combined_new_filtering_mprasnakeflow_count_table['name'] != 'no_BC']['name'].nunique() # 70788

67832

#### Checking the number of oligos found 

last time: 

In [44]:
no_resequencing_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/experiments/standard_bwa/assigned_counts/assignmentFixDuplicates/standardConfig/NGN2_allreps_minThreshold_merged.tsv.gz'
no_resequencing_no_threshold_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/experiments/standard_bwa/assigned_counts/assignmentFixDuplicates/standardConfig/NGN2_allreps_merged.tsv.gz'
no_resequencing_count_table = pd.read_csv(no_resequencing_path, sep="\t")

In [45]:
no_resequencing_count_table.head()
no_resequencing_count_table = no_resequencing_count_table.loc[no_resequencing_count_table['name'] != 'no_BC']

In [46]:
no_resequencing_count_table.groupby('replicate').count()


,condition,name,dna_counts,rna_counts,dna_normalized,rna_normalized,ratio,log2,n_obs_bc
replicate,,,,,,,,,
1,67883,67883,67883,67883,67883,67883,67883,67883,67883
2,67848,67848,67848,67848,67848,67848,67848,67848,67848
3,67861,67861,67861,67861,67861,67861,67861,67861,67861


In [58]:
# merge the sequences over the replicates
total_dna_counts = no_resequencing_count_table[["dna_counts"]].sum().iloc[0]
total_rna_counts = no_resequencing_count_table[["rna_counts"]].sum().iloc[0]
combined_no_resequencing_count_table = combine_replicates(no_resequencing_count_table, total_dna_counts, total_rna_counts)
combined_no_resequencing_count_table = combined_no_resequencing_count_table.loc[combined_no_resequencing_count_table['replicates'] == 3]

print(combined_no_resequencing_count_table['name'].nunique()/ 80215)
combined_no_resequencing_count_table['name'].nunique()

0.8343077977934301


66924

In [62]:
combined_no_resequencing_count_table['label'] = combined_no_resequencing_count_table['name'].apply(hf.get_label)
print(66924 - 62081)
print((66924 - 62081)/66924)
print((62081)/66924)
combined_no_resequencing_count_table['label'].value_counts()

4843
0.07236566851951468
0.9276343314804854


label
cardiac_neuro_cava_random            62081
MK                                    1950
C_positive_heart_AB                    545
GC_Selvarajan                          309
GC_Vista                               216
GC_Mendelian_variants                  192
GC_Cort_Chengyu                        185
C_negative_heart_MK                    176
C_negative_neuron_MK                   167
C_SLEA                                 160
GC_Kircher                             155
C_negative_neuron_NP                   101
C_positive_neuron_MK                    93
C_positive_heart_CAD                    84
C_positive_heart_MK                     70
GC_GABA_Chengyu                         65
C_positive_neuron_NP                    54
GC_DNase_positive_shuffeled             51
C_positive_neuron_CD                    50
GC_Atrial_fib                           43
GC_DNase_positive                       37
GC_Glut_Chengyu                         31
GC_Mohlke                               29
GC_DN

In [65]:
combined_no_resequencing_count_table

,condition,name,replicates,dna_counts,rna_counts,dna_normalized,rna_normalized,ratio,log2,mean_dna_counts,mean_rna_counts,mean_dna_normalized,mean_rna_normalized,mean_ratio,mean_log2,mean_n_obs_bc,label
0,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3,1069,2587,26.282910,27.005356,1.027487,0.039121,356.333333,862.333333,0.121140,0.126572,1.080802,0.111316,108,C_SLEA
1,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3,1140,5188,28.028547,54.156857,1.932203,0.950247,380.000000,1729.333333,0.137726,0.270879,2.040802,1.022238,101,C_SLEA
2,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,3,638,1632,15.686152,17.036236,1.086069,0.119115,212.666667,544.000000,0.116224,0.128225,1.140693,0.189335,67,C_SLEA
3,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,3,922,2260,22.668702,23.591846,1.040723,0.057586,307.333333,753.333333,0.124520,0.132123,1.099769,0.132576,90,C_SLEA
4,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,3,595,1412,14.628935,14.739684,1.007571,0.010881,198.333333,470.666667,0.133440,0.136954,1.060619,0.084120,54,C_SLEA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68727,NGN2,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,3,98,110,2.409472,1.148276,0.476567,-1.069248,32.666667,36.666667,0.082304,0.038923,0.493710,-1.040323,14,cardiac_neuro_cava_random
68728,NGN2,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,3,631,2799,15.514047,29.218397,1.883351,0.913302,210.333333,933.000000,0.102176,0.195736,1.986678,0.982698,75,cardiac_neuro_cava_random
68729,NGN2,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,3,197,646,4.843530,6.743510,1.392272,0.477441,65.666667,215.333333,0.099126,0.140391,1.475498,0.558521,24,cardiac_neuro_cava_random
68730,NGN2,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,3,216,334,5.310672,3.486583,0.656524,-0.607081,72.000000,111.333333,0.089462,0.059643,0.690325,-0.539574,29,cardiac_neuro_cava_random


In [68]:
# investigate the matching problem
# no_resequencing_meta_count_data = metadata_file.merge(combined_no_resequencing_count_table, right_on='name', left_on="header")

0.12606111370628695

In [72]:
no_resequencing_count_metadata = combined_no_resequencing_count_table.merge(metadata_file, left_on='name', right_on="header", how='left')
hopefully_non = no_resequencing_count_metadata.loc[no_resequencing_count_metadata['category'].isna()]
hopefully_non # []
no_resequencing_count_metadata_merged = no_resequencing_count_metadata.loc[~no_resequencing_count_metadata['category'].isna()]
no_resequencing_count_metadata_merged #66857  
no_resequencing_count_metadata_merged.groupby('class')['category'].value_counts()

class                     category 
element active control    element        197
element inactive control  element       3868
                          scrambled      551
                          synthetic      160
test                      variant      54255
                          element       7826
Name: count, dtype: int64

In [ ]:
7826/62081

In [74]:
no_resequencing_count_metadata_merged.groupby('class')['allele'].value_counts()

class  allele
test   alt       38573
       ref       15682
Name: count, dtype: int64

In [77]:
15682/62081
38573/62081

0.6213334192426024

with new filtering

In [17]:
new_filtering_mprasnakeflow_count_table.groupby('replicate').count()

,condition,name,dna_counts,rna_counts,dna_normalized,rna_normalized,ratio,log2,n_obs_bc
replicate,,,,,,,,,
1,68567,68567,68567,68567,68567,68567,68567,68567,68567
2,68537,68537,68537,68537,68537,68537,68537,68537,68537
3,68553,68553,68553,68553,68553,68553,68553,68553,68553


In [56]:
combined_new_filtering_mprasnakeflow_count_table['name'].nunique()
67832/ 80215

0.8456273764258555

In [51]:
combined_new_filtering_mprasnakeflow_count_table['replicates'].value_counts()

replicates
3    67832
2      784
1      590
Name: count, dtype: int64

#### Merge with the duplicated fasta
- If it is merged with the duplicated design we can use the metadata file and make just a few simple panda commands for many numbers
- remove on both sides the not so interesting sequences and check then how many can be found


In [24]:
metadata_path = "/home/kisa/coding/80K_MPRA/design_data/metadata_example_2504.tsv"
metadata_file = pd.read_csv(metadata_path, sep="\t")
metadata_file.head()

,header,sequence,tmp_label,name,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,tmp_matching_header
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....


In [29]:
new_filter_count_metadata = combined_new_filtering_mprasnakeflow_count_table.merge(metadata_file, left_on='name', right_on="header")
# hopefully_non = new_filter_count_metadata.loc[new_filter_count_metadata['category'].isna()]
# hopefully_non # []
new_filter_count_metadata_merged = new_filter_count_metadata.loc[~new_filter_count_metadata['category'].isna()]
new_filter_count_metadata_merged # 69139 
new_filter_count_metadata_merged.category.value_counts()

category
variant      56104
element      12309
scrambled      564
synthetic      162
Name: count, dtype: int64

In [34]:
print('Controls: ', 69139 - 64175,  (69139 - 64175)/ 69139)
print('tested: ', 64175/69139)
new_filter_count_metadata_merged.tmp_label.value_counts() # 64175

Controls:  4964 0.07179739365625769
tested:  0.9282026063437423


tmp_label
cardiac_neuro_cava_random            64175
MK                                    2003
C_positive_heart_AB                    576
GC_Selvarajan                          324
GC_Vista                               223
GC_Mendelian_variants                  195
C_negative_heart_MK                    194
GC_Cort_Chengyu                        185
C_negative_neuron_MK                   176
C_SLEA                                 162
GC_Kircher                             159
C_negative_neuron_NP                   121
C_positive_neuron_MK                    93
C_positive_heart_CAD                    86
C_positive_heart_MK                     72
GC_GABA_Chengyu                         69
C_positive_neuron_NP                    63
C_positive_neuron_CD                    53
GC_DNase_positive_shuffeled             51
GC_Atrial_fib                           44
GC_Glut_Chengyu                         31
GC_Mohlke                               30
GC_DNase_negative_blood_shuffeled       18
G

In [37]:
new_filter_count_metadata_merged.groupby('class')['category'].value_counts()

class                     category 
element active control    element        209
element inactive control  element       4029
                          scrambled      564
                          synthetic      162
test                      variant      56104
                          element       8071
Name: count, dtype: int64

In [39]:
new_filter_count_metadata_merged.groupby('class')['allele'].value_counts()

class  allele
test   alt       39903
       ref       16201
Name: count, dtype: int64

In [43]:
8071/64175
39903/64175
16201/64175

0.2524503311258278

In [22]:
def add_matching_header(header):
    """gets header with > and , and replaces it with * and ~"""
    header = header.replace('*', '>')
    header = header.replace('~', ',')
    return header

def is_dnase_header(header):
    """If DNase in header"""
    return 'dnase' in header.lower()

def is_interesting_header(label):
    """Check if label is in the interesting group list"""
    groups_of_interest = config['analysis_info']['groups_of_interest']
    return label in groups_of_interest


In [6]:
design_fa['matching_header'] = design_fa['header'].apply(add_matching_header)
design_fa.head()


,header,sequence,label,matching_header
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:SKI|ENSG00000157933....


In [8]:
# combined_new_filtering_mprasnakeflow_count_table.head()
combined_mprasnakeflow_count_table.head()

,condition,name,replicates,dna_counts,rna_counts,dna_normalized,rna_normalized,ratio,log2,mean_dna_counts,mean_rna_counts,mean_dna_normalized,mean_rna_normalized,mean_ratio,mean_log2,mean_n_obs_bc
0,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3,1575,3861,14.898326,15.323954,1.028569,0.040638,525.000000,1287.000000,0.115009,0.118266,1.064153,0.088407,129
1,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3,1599,7107,15.125348,28.207030,1.864885,0.899086,533.000000,2369.000000,0.132086,0.246448,1.939186,0.945874,114
2,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,3,935,2412,8.844403,9.573006,1.082380,0.114207,311.666667,804.000000,0.101674,0.109971,1.124449,0.164545,87
3,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,3,1351,3324,12.779453,13.192651,1.032333,0.045908,450.333333,1108.000000,0.116685,0.120833,1.074045,0.099668,109
4,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,3,811,1999,7.671455,7.933847,1.034204,0.048520,270.333333,666.333333,0.122675,0.127063,1.070345,0.097955,62


In [9]:
print(f'Number of MPRAsnakeflow results: {combined_mprasnakeflow_count_table.shape[0]}')

Number of MPRAsnakeflow results: 70789


In [36]:
matched_combined_design_fa = combined_new_filtering_mprasnakeflow_count_table.merge(design_fa, left_on="name", right_on="matching_header")
all_mprasnakeflow = set(combined_new_filtering_mprasnakeflow_count_table['name'].to_list())
all_designed = set(design_fa['matching_header'].to_list())
matched_combined_design_fa_not_matchable = all_designed - all_mprasnakeflow
print('Number of not matchable: ', len(matched_combined_design_fa_not_matchable)) # 12446

Number of not matchable:  12446


In [20]:
matched_combined_design_fa = combined_mprasnakeflow_count_table.merge(design_fa, left_on="name", right_on="header")
print(combined_mprasnakeflow_count_table.shape[0], matched_combined_design_fa.shape[0])
matched_mprasnakeflow = set(matched_combined_design_fa['name'].to_list())
mprasnakeflow_results = set(combined_mprasnakeflow_count_table['name'].to_list())
missing_from_design = mprasnakeflow_results-matched_mprasnakeflow
missing_from_design

70789 70720


{'GC_DNase_negative_blood:chr10:16877280-16877549_active_count_12_3_._',
 'GC_DNase_negative_blood:chr12:4360422-4360691_active_count_12_2_._',
 'GC_DNase_negative_blood:chr14:78017728-78017997_active_count_12_3_._',
 'GC_DNase_negative_blood:chr17:14829903-14830172_active_count_12_2_._',
 'GC_DNase_negative_blood:chr18:29321172-29321441_active_count_13_3_._',
 'GC_DNase_negative_blood:chr19:40349220-40349489_active_count_12_2_._',
 'GC_DNase_negative_blood:chr1:28199346-28199615_active_count_12_3_._',
 'GC_DNase_negative_blood:chr1:8213090-8213359_active_count_12_1_._',
 'GC_DNase_negative_blood:chr20:48225489-48225758_active_count_12_3_._',
 'GC_DNase_negative_blood:chr3:136110531-136110800_active_count_13_3_._',
 'GC_DNase_negative_blood:chr5:61592889-61593158_active_count_12_3_._',
 'GC_DNase_negative_blood:chr6:150400882-150401151_active_count_14_3_._',
 'GC_DNase_negative_blood:chr6:33501163-33501432_active_count_12_3_._',
 'GC_DNase_negative_blood:chr7:156715928-156716197_active

In [ ]:
all_mprasnakeflow = set(combined_mprasnakeflow_count_table['name'].to_list())
all_designed = set(design_fa['header'].to_list())
matched_combined_design_fa_not_matchable = all_designed - all_mprasnakeflow
print('Number of not matchable: ', len(matched_combined_design_fa_not_matchable)) # 12446

In [14]:
matched_combined_design_fa_not_matchable

{'cardiac_neuro_cava_random:ALT_SETD1A|ENSG00000099381.19|EH38E3177291_fwd_tile1-1_SETD1A|ENSG00000099381.19|EH38E3177291|16-30928266-G-A',
 'cardiac_neuro_cava_random:REF_MYH6|ENSG00000197616.13|EH38E1703157,MYH7|ENSG00000092054.13|EH38E1703157_rev_tile1-1',
 'cardiac_neuro_cava_random:ALT_B3GNT9|ENSG00000237172.4|EH38E3186345_rev_tile1-1_B3GNT9|ENSG00000237172.4|EH38E3186345|16-67141686-C-T',
 'cardiac_neuro_cava_random:ALT_CDKN2A|ENSG00000147889.18|EH38E3878012,CDKN2B|ENSG00000147883.12|EH38E3878012_rev_tile1-1_CDKN2A|ENSG00000147889.18|EH38E3878012|9-22007331-C-A,CDKN2B|ENSG00000147883.12|EH38E3878012|9-22007331-C-A',
 'C_positive_heart_AB:ARHGAP31-ENST00000482743.1:Oligo-Sub-Id:0.2:chr3:119322940-119323210:Length::270',
 'C_positive_heart_AB:ROR2-ENST00000495386.5:Oligo-Sub-Id:0.1:chr9:91950228-91950498:Length::270',
 'GC_Selvarajan:ALT_rs2625259|STARR-seq-HepG2,rs6824003|STARR-seq-HepG2_fwd_tile2-3_rs6824003',
 'cardiac_neuro_cava_random:ALT_CYP2D6|ENSG00000100197.23|EH38E2166940

In [35]:
combined_new_filtering_mprasnakeflow_count_table.loc[combined_new_filtering_mprasnakeflow_count_table['name'] == 'cardiac_neuro_cava_random:ALT_MYBPC3|ENSG00000134571.12|EH38E2957409_rev_tile1-1_MYBPC3|ENSG00000134571.12|EH38E2957409|11-47337392-G-A']

,condition,name,replicates,dna_counts,rna_counts,dna_normalized,rna_normalized,ratio,log2,mean_dna_counts,mean_rna_counts,mean_dna_normalized,mean_rna_normalized,mean_ratio,mean_log2,mean_n_obs_bc


#### Investigate group numbers
- Make table for design and for mprasnakeflow result

In [42]:
# design df label counts
design_numbers = design_fa['label'].value_counts().rename_axis('group_names').reset_index(name='design_counts')

,condition,replicate,name,dna_counts,rna_counts,dna_normalized,rna_normalized,ratio,log2,n_obs_bc
0,NGN2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,352,882,0.124860,0.125379,1.039527,0.055928,106
1,NGN2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,340,1902,0.129131,0.289492,2.320821,1.214635,99
2,NGN2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,219,598,0.121094,0.132511,1.132836,0.179939,68
3,NGN2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,276,809,0.116602,0.136968,1.216045,0.282196,89
4,NGN2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,191,467,0.132993,0.130312,1.014363,0.020575,54
...,...,...,...,...,...,...,...,...,...,...
203590,NGN2,3,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,217,968,0.104597,0.206934,2.066972,1.047519,75
203591,NGN2,3,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,78,230,0.108453,0.141832,1.366319,0.450295,26
203592,NGN2,3,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,69,100,0.086015,0.055287,0.671537,-0.574462,29
203593,NGN2,3,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,342,655,0.118881,0.100978,0.887430,-0.172295,104


In [46]:
mprasnakeflow_count_table = pd.read_csv(config['files']['creating']['mprasnakeflow_count_minthreshold'], sep="\t")
mprasnakeflow_count_table
mprasnakeflow_count_table = mprasnakeflow_count_table.loc[mprasnakeflow_count_table['name'] != 'no_BC']


# combine replicates
# merge the sequences over the replicates
total_dna_counts = mprasnakeflow_count_table[["dna_counts"]].sum().iloc[0]
total_rna_counts = mprasnakeflow_count_table[["rna_counts"]].sum().iloc[0]

combined_mprasnakeflow_count_table = combine_replicates(mprasnakeflow_count_table, total_dna_counts, total_rna_counts)
# filter counts
min_15bcs_mprasnakeflow = mprasnakeflow_count_table.loc[mprasnakeflow_count_table['n_obs_bc'] >= 15]
min_15bcs_combined_mprasnakeflow_count_table = combine_replicates(min_15bcs_mprasnakeflow, total_dna_counts, total_rna_counts)
min_15bcs_combined_mprasnakeflow_count_table['label'] = min_15bcs_combined_mprasnakeflow_count_table['name'].apply(hf.get_label)

combined_mprasnakeflow_count_table['label'] = combined_mprasnakeflow_count_table['name'].apply(hf.get_label)
mprasnakeflow_label_numbers = combined_mprasnakeflow_count_table['label'].value_counts().rename_axis('group_names').reset_index(name='MPRAsnakeflow_counts')
mprasnakeflow_15bcs_label_numbers = min_15bcs_combined_mprasnakeflow_count_table['label'].value_counts().rename_axis('group_names').reset_index(name='MPRAsnakeflow_counts')


In [47]:
mprasnakeflow_15bcs_label_numbers

,group_names,MPRAsnakeflow_counts
0,cardiac_neuro_cava_random,60226
1,MK,1874
2,C_positive_heart_AB,519
3,GC_Selvarajan,302
4,GC_Vista,207
5,GC_Mendelian_variants,188
6,GC_Cort_Chengyu,185
7,C_negative_heart_MK,165
8,C_negative_neuron_MK,160
9,C_SLEA,159


merge them

In [48]:
design_mprasnakeflow_numbers = design_numbers.merge(mprasnakeflow_label_numbers, on='group_names')
design_mprasnakeflow_numbers.merge(mprasnakeflow_15bcs_label_numbers, on='group_names')

,group_names,design_counts,MPRAsnakeflow_counts_x,MPRAsnakeflow_counts_y
0,cardiac_neuro_cava_random,73940,63737,60226
1,MK,2397,1998,1874
2,C_positive_heart_AB,909,573,519
3,GC_Selvarajan,364,320,302
4,GC_Vista,256,222,207
5,C_negative_heart_MK,243,188,165
6,C_negative_neuron_MK,222,175,160
7,C_negative_neuron_NP,217,116,97
8,GC_Mendelian_variants,209,194,188
9,GC_Kircher,203,155,152


#### Defining groups of focus:
cardiac_neuro_cava_random, MK, NP, CD (neuro focus)

In [17]:
# groups_of_interest = ["cardiac_neuro_cava_random", "MK", "C_positive_neuron_NP", "C_positive_neuron_MK", "C_negative_neuron_MK", "C_negative_neuron_NP", "C_positive_neuron_CD"]
groups_of_interest = config['analysis_info']['groups_of_interest']


['cardiac_neuro_cava_random',
 'MK',
 'C_positive_neuron_NP',
 'C_positive_neuron_MK',
 'C_negative_neuron_MK',
 'C_negative_neuron_NP',
 'C_positive_neuron_CD']

In [36]:
# compute the interesting number and then the missings:
print(f'All interesting sequences: ', design_fa['label'].apply(is_interesting_header).sum())
print(f'All sequences: ', design_fa['header'].nunique())
combined_mprasnakeflow_count_table['label'] = combined_mprasnakeflow_count_table['name'].apply(hf.get_label)
combined_new_filtering_mprasnakeflow_count_table['label'] = combined_mprasnakeflow_count_table['name'].apply(hf.get_label)
combined_mprasnakeflow_count_table.head()
print(f'All interesting and found sequences: ', combined_mprasnakeflow_count_table['label'].apply(is_interesting_header).sum())
print(f'All interesting and found sequences: new filtering ', combined_new_filtering_mprasnakeflow_count_table['label'].apply(is_interesting_header).sum())
print(f'Found ratio: (intersting) ', combined_mprasnakeflow_count_table['label'].apply(is_interesting_header).sum() / design_fa['label'].apply(is_interesting_header).sum())
print(f'Found ratio: (intersting) new filtering', combined_new_filtering_mprasnakeflow_count_table['label'].apply(is_interesting_header).sum() / design_fa['label'].apply(is_interesting_header).sum())

All interesting sequences:  77065
All sequences:  80215
All interesting and found sequences:  68175
All interesting and found sequences: new filtering  66593
Found ratio: (intersting)  0.8846428339713229
Found ratio: (intersting) new filtering 0.8641147083630701


In [32]:
print(f'Found ratio: ', combined_mprasnakeflow_count_table['name'].nunique() / design_fa['header'].nunique())
print(f'Found ratio: ', combined_new_filtering_mprasnakeflow_count_table['name'].nunique() / design_fa['header'].nunique())


Found ratio:  0.8824908059589852
Found ratio:  0.8627563423299881


#### Investigate numbers of REF and ALT and elements more specifically


In [ ]:
# TODO

#### Plot the controls nicely

In [78]:
new_filtering_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/resequencing_results/new_bam_filtering/results/experiments/standard_bwa/assigned_counts/assignmentFixDuplicates/standardConfig/NGN2_allreps_minThreshold_merged.tsv.gz'

new_filtering_mprasnakeflow_count_table = pd.read_csv(new_filtering_path, sep="\t")
new_filtering_mprasnakeflow_count_table = new_filtering_mprasnakeflow_count_table.loc[new_filtering_mprasnakeflow_count_table['name'] != 'no_BC']
# merge the sequences over the replicates
total_dna_counts = new_filtering_mprasnakeflow_count_table[["dna_counts"]].sum().iloc[0]
total_rna_counts = new_filtering_mprasnakeflow_count_table[["rna_counts"]].sum().iloc[0]
combined_new_filtering_mprasnakeflow_count_table = combine_replicates(new_filtering_mprasnakeflow_count_table, total_dna_counts, total_rna_counts)
combined_new_filtering_mprasnakeflow_count_table = combined_new_filtering_mprasnakeflow_count_table.loc[combined_new_filtering_mprasnakeflow_count_table['replicates'] == 3]


In [80]:
combined_new_filtering_mprasnakeflow_count_table['label'] = combined_new_filtering_mprasnakeflow_count_table['name'].apply(hf.get_label)

In [81]:
combined_new_filtering_mprasnakeflow_count_table.head()

,condition,name,replicates,dna_counts,rna_counts,dna_normalized,rna_normalized,ratio,log2,mean_dna_counts,mean_rna_counts,mean_dna_normalized,mean_rna_normalized,mean_ratio,mean_log2,mean_n_obs_bc,label
0,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3,1359,3366,25.783429,26.411973,1.024378,0.034748,453.000000,1122.000000,0.112556,0.117015,1.074407,0.102679,114,C_SLEA
1,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3,1458,6650,27.661692,52.180517,1.886382,0.915622,486.000000,2216.666667,0.132854,0.254553,1.989397,0.983514,103,C_SLEA
2,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,3,824,2161,15.633219,16.956706,1.084659,0.117241,274.666667,720.333333,0.107339,0.117987,1.140071,0.186331,72,C_SLEA
3,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,3,1198,2928,22.728880,22.975121,1.010834,0.015546,399.333333,976.000000,0.117521,0.120830,1.065769,0.088221,96,C_SLEA
4,NGN2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,3,742,1840,14.077486,14.437917,1.025603,0.036473,247.333333,613.333333,0.124235,0.129395,1.075619,0.105127,56,C_SLEA


In [ ]:
# function for plotting a specific list of labels as violin plot
def plot_subgroup_violin(labels_list, save_path=None, alternative_labels=None, colors=None):
    """Plot a violin plot just for a subset of labels
    @colors: dict of colors for each label
    """
    subset_rep_label_df = count_rep1_label[count_rep1_label["label"].isin(labels_list)]
    avg_seq_per_label = int(subset_rep_label_df.groupby("label").count()["name"].mean())

    fig, ax = plt.subplots(figsize=(10,8))
    ax.set_title(f'Log2 for {len(labels_list)} labels and average number of sequences per label is {avg_seq_per_label}')
    if colors:
        sns.violinplot(x=subset_rep_label_df["label"], y=subset_rep_label_df["log2"], hue=subset_rep_label_df["label"], ax=ax, palette=colors)
    else:
        sns.violinplot(x=subset_rep_label_df["label"], y=subset_rep_label_df["log2"], hue=subset_rep_label_df["label"], ax=ax)

    if alternative_labels:
        # add alternative labels
        ax.set_xticklabels(alternative_labels)
    plt.xticks(rotation=90)
    ax.set_xlabel('Label')
    ax.set_ylabel('log2(RNA/DNA)')
    # add line at 0 
    ax.axhline(y=0, color='black', linestyle='--')
    if save_path:
        plt.savefig(save_path, dpi=600, bbox_inches='tight') # bbox_inches='tight' to prevent cutting off labels

plot_subgroup_violin(['GC_DNase_negative_blood', 'GC_DNase_negative_blood_shuffeled',
       'GC_DNase_negative_brain', 'GC_DNase_negative_brain_shuffeled',
       'GC_DNase_positive', 'GC_DNase_positive_shuffeled', 'C_negative_neuron_NP', 'C_positive_neuron_MK', 'C_positive_neuron_NP', 'C_negative_neuron_MK'], "../results/control_plots/shuffeled_vs_not_shuffeled_controls.png", None, None)

# for keeping the colors
color_dict = {'GC_DNase_negative_blood': sns.color_palette()[0+4], 'GC_DNase_negative_blood_shuffeled': sns.color_palette()[1+4], 'GC_DNase_negative_brain': sns.color_palette()[2+4], 'GC_DNase_negative_brain_shuffeled': sns.color_palette()[3+4], 'GC_DNase_positive': sns.color_palette()[4+4], 'GC_DNase_positive_shuffeled': sns.color_palette()[5+4]}

# all DNase from encode
plot_subgroup_violin(['GC_DNase_positive', 'GC_DNase_positive_shuffeled'], "../results/control_plots/GC_DNase_positive_vs_shuffeled.png", None, color_dict)

# all from blood (negative for us)
plot_subgroup_violin(['GC_DNase_negative_blood', 'GC_DNase_negative_blood_shuffeled'], "../results/control_plots/GC_DNase_negative_blood_vs_shuffeled.png", None, color_dict)

# all from brain (for 80K positive and shuffeled: negative)
plot_subgroup_violin(['GC_DNase_negative_brain', 'GC_DNase_negative_brain_shuffeled'], "../results/control_plots/GC_DNase_negative_brain_vs_shuffeled.png", ['GC_DNase_brain', 'GC_DNase_brain_shuffeled'], color_dict)
# all from brain (for 80K positive and negative)
plot_subgroup_violin(['GC_DNase_negative_brain', 'GC_DNase_negative_brain_shuffeled'], "../results/control_plots/GC_DNase_negative_brain_vs_shuffeled.png", ['GC_DNase_brain', 'GC_DNase_brain_shuffeled'], color_dict)
